In [2]:
# train_models_structured.py
import re
import os
import json
import itertools
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from utils import MLPTextGenerator, train_model

# --- Tokenizer (Identical to your script) ---
TOKEN_REGEX = re.compile(r"""
    (\\[a-zA-Z]+) |                  # LaTeX commands
    (::|->|==|!=|<=|>=|&&|\|\||<<|>>|\+=|-=|\*=|/=|%=|&=|\^=|\|=) |  # Multi-char ops
    ([a-zA-Z_][a-zA-Z0-9_]*) |          # Identifiers
    (\d+\.\d*|\.\d+|\d+) |              # Numbers
    (\S)                               # Other non-whitespace chars
""", re.VERBOSE)

def tokenize_code(line): 
    return [m.group(0) for m in TOKEN_REGEX.finditer(line)]

# --- PREPROCESSING FUNCTIONS (Refactored for looping) ---
def load_and_process_text_structured():
    """
    Loads all text data and builds the complete vocabulary.
    This is run only once.
    """
    files = [
        "Data2/stacks-project-data.txt",
        # "Data2/temp.txt"
             ]
    all_words = []
    print("Loading and processing structured text data (this will run only once)...")

    for path in files:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line: 
                        all_words.extend(tokenize_code(line))
        except FileNotFoundError:
            print(f"Warning: File not found {path}. Skipping.")

    vocab = sorted(list(set(all_words)))
    word_to_ix = {w: i for i, w in enumerate(vocab)}
    ix_to_word = {i: w for i, w in enumerate(vocab)}

    print(f"Total tokens: {len(all_words)} | Vocabulary size: {len(vocab)}")
    cnt = Counter(all_words)
    print("Top 10:", cnt.most_common(10))
    print("Bottom 10:", cnt.most_common()[:-11:-1])
    
    return all_words, word_to_ix, ix_to_word

def create_training_data_structured(all_words, word_to_ix, CONTEXT_SIZE, TEST_SPLIT, BATCH_SIZE):
    """
    Generates X/y pairs and DataLoaders for a *specific* CONTEXT_SIZE.
    """
    print(f"Creating training data for CONTEXT_SIZE={CONTEXT_SIZE}...")
    X, y = [], []
    for i in range(len(all_words) - CONTEXT_SIZE):
        X.append([word_to_ix[w] for w in all_words[i:i+CONTEXT_SIZE]])
        y.append(word_to_ix[all_words[i+CONTEXT_SIZE]])
        
    print(f"Data pairs created. X shape: ({len(X)}, {len(X[0])}), y shape: ({len(y)})")

    # Convert to tensors
    X_tensor = torch.tensor(X, dtype=torch.long)
    y_tensor = torch.tensor(y, dtype=torch.long)
    
    # Create dataset
    dataset = TensorDataset(X_tensor, y_tensor)
    val_size = int(len(dataset) * TEST_SPLIT)
    train_size = len(dataset) - val_size
    
    # Use sequential split (as in your original code)
    train_dataset = TensorDataset(X_tensor[:train_size], y_tensor[:train_size])
    val_dataset = TensorDataset(X_tensor[train_size:], y_tensor[train_size:])
    
    # Create DataLoaders
    # Using num_workers=2 and pin_memory=True as in your original
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    print(f"Created data loaders. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    return train_loader, val_loader

# --- MAIN EXECUTION BLOCK (Modified to train all 36 models) ---
if __name__ == "__main__":

    # --- Hyperparameters (from your original script) ---
    HIDDEN_DIM = 1024       
    LEARNING_RATE = 0.001
    EPOCHS = 50             
    BATCH_SIZE = 1024
    TEST_SPLIT = 0.1
    SEED = 43

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Global device set to: {device}")

    # --- Define Model Variant *Options* ---
    context_sizes = [5, 10, 15]
    embedding_dims = [32, 64]
    num_hidden_layers_list = [1, 2, 3]
    activations = ['tanh']

    # --- Generate All Combinations ---
    print("Generating all model configurations...")
    model_configs = []
    all_combinations = itertools.product(
        context_sizes, 
        embedding_dims, 
        num_hidden_layers_list, 
        activations
    )

    for (cs, ed, hl, act) in all_combinations:
        act_name = "Relu" if act == "relu" else "Tanh"
        name = f"CS{cs}_ED{ed}_HL{hl}_{act_name}"
        config = {
            "name": name,
            "CONTEXT_SIZE": cs,
            "EMBEDDING_DIM": ed,
            "NUM_HIDDEN_LAYERS": hl,
            "ACTIVATION": act,
        }
        model_configs.append(config)

    print(f"Generated {len(model_configs)} model configurations to train.")

    # --- Setup ---
    # Save to a new directory
    output_dir = "Models_Structured"
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Load data and build vocab ONCE
    all_words, word_to_ix, ix_to_word = load_and_process_text_structured()
    VOCAB_SIZE = len(word_to_ix)
    
    # 2. Save the vocabulary ONCE
    vocab_path = os.path.join(output_dir, "vocab.json")
    with open(vocab_path, 'w') as f:
        json.dump({'word_to_ix': word_to_ix, 'ix_to_word': ix_to_word}, f)
    print(f"Vocabulary saved to {vocab_path}\n")

    # 3. Loop, Train, and Save each model variant
    total_configs = len(model_configs)
    for i, config in enumerate(model_configs):
        print(f"\n--- Training Model {i+1}/{total_configs}: {config['name']} ---")
        
        # A. Create data loaders for this config's CONTEXT_SIZE
        train_loader, val_loader = create_training_data_structured(
            all_words, 
            word_to_ix, 
            config['CONTEXT_SIZE'], 
            TEST_SPLIT, 
            BATCH_SIZE
        )
        
        # B. Define model hyperparameters
        model_params = {
            'VOCAB_SIZE': VOCAB_SIZE,
            'EMBEDDING_DIM': config['EMBEDDING_DIM'],
            'CONTEXT_SIZE': config['CONTEXT_SIZE'],
            'NUM_HIDDEN_LAYERS': config['NUM_HIDDEN_LAYERS'],
            'HIDDEN_LAYER_DIM': HIDDEN_DIM,
            'ACTIVATION_TYPE': config['ACTIVATION'],
        }

        # C. Initialize Model
        model = MLPTextGenerator(**model_params)
        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
            model = nn.DataParallel(model)
        
        # D. Train Model
        history, trained_model = train_model(
            model, 
            train_loader, 
            val_loader, 
            LEARNING_RATE, 
            EPOCHS, 
            device
        )
        
        # E. Save the model
        save_path = os.path.join(output_dir, f"{config['name']}.pth")
        
        torch.save({
            'hyperparameters': model_params,
            'model_state_dict': trained_model.state_dict(),
        }, save_path)
        
        print(f"Model {i+1}/{total_configs} ('{config['name']}') saved to {save_path}")
        print("-" * (30 + len(config['name'])) + "\n")

    print(f"--- ALL {total_configs} MODELS TRAINED AND SAVED ---")

Global device set to: cuda
Generating all model configurations...
Generated 18 model configurations to train.
Loading and processing structured text data (this will run only once)...
Total tokens: 8468301 | Vocabulary size: 12997
Top 10: [('$', 904830), ('{', 486749), ('}', 486648), ('-', 337705), (',', 199511), ('.', 199003), ('(', 191818), (')', 191792), ('\\mathcal', 131698), ('the', 124244)]
Bottom 10: [('converts', 1), ('Missing', 1), ('equidmensional', 1), ('equdimensional', 1), ('supplementary', 1), ('Deails', 1), ('coclude', 1), ('hypothese', 1), ('Correspondences', 1), ('Equipped', 1)]
Vocabulary saved to Models_Structured\vocab.json


--- Training Model 1/18: CS5_ED32_HL1_Tanh ---
Creating training data for CONTEXT_SIZE=5...
Data pairs created. X shape: (8468296, 5), y shape: (8468296)
Created data loaders. Train batches: 7443, Val batches: 827
Starting training on cuda for up to 50 epochs...
  [Epoch 1] Batch 200/7443 processed. (7243 left)
  [Epoch 1] Batch 400/7443 process

KeyboardInterrupt: 